# Attention mechanism

Attention 메커니즘은 RNN(Recurrent Neural Network) 및 LSTM, GRU와 같은 순환 신경망 모델이 가지고 있는 여러 문제점을 해결하기 위해 제안된 기술이다. 특히 자연어 처리(NLP) 및 시계열 데이터 처리와 같은 분야에서 매우 중요한 역할을 한다.

## 1. 기존 RNN의 문제점

### 1. 긴 시퀀스 학습의 어려움 (Long-Term Dependencies 문제)

- RNN은 이전 단계의 은닉 상태를 다음 단계로 전달하며 순차적으로 데이터를 처리한다.
- 시퀀스가 길어질수록 초반 입력 정보는 시간이 지나면서 사라지거나 희석되기 때문에 장기 의존성(Long-Term Dependency)을 학습하기 어렵다.
- 이는 기울기 소실(Vanishing Gradient) 문제로도 이어진다.

### 2. 정보 병목 현상 (Information Bottleneck)

- RNN은 고정된 크기의 **컨텍스트 벡터(Context Vector)** 를 사용해 입력 시퀀스 전체 정보를 인코딩한다.
- 입력 길이가 길어질수록 중요한 정보를 완벽히 담아내지 못하며, 출력에 필요한 세부 정보가 손실될 가능성이 크다.

### 3. 병렬 처리 불가 (Lack of Parallelism)

- RNN은 순차적으로 데이터를 처리해야 하기 때문에 병렬 처리가 어렵다.
- 이는 모델 학습 속도를 느리게 만들고, 대규모 데이터 처리에 비효율적이다.

### 4. 기억력의 한계 (Capacity Issue)

- 긴 문장이나 복잡한 입력 데이터를 처리할 때, 단일 컨텍스트 벡터가 모든 중요한 정보를 저장하기에는 용량(capacity)의 한계가 있다.


## 2. Attention 메커니즘의 원리


![https://www.tensorflow.org/text/tutorials/transformer?why_transformers_are_significant](https://d.pr/i/O6WlEu+)

https://www.tensorflow.org/text/tutorials/transformer?why_transformers_are_significant

Attention 메커니즘은 입력 시퀀스의 모든 요소를 동일하게 처리하는 대신, 중요한 부분에 가중치를 부여하여 더욱 효율적으로 정보를 처리하는 방법이다.

이를 통해 모델은 특정 단어(또는 시점)에 더 많은 관심을 기울이며 필요한 정보를 선택적으로 사용할 수 있다.

### 1. 가중치 할당 (Attention Weights)

- 입력 시퀀스의 각 요소에 대해 얼마나 중요한지를 나타내는 가중치를 계산한다.
- 이 가중치는 쿼리(Query), 키(Key), 값(Value)의 연산을 통해 얻어진다.

### 2. 동적 컨텍스트 생성

- RNN처럼 고정된 크기의 컨텍스트 벡터를 사용하는 대신,
- Attention 메커니즘은 가중치가 적용된 입력의 가중합(Weighted Sum)을 통해 동적으로 컨텍스트 벡터를 생성한다.
- 이는 출력 시퀀스의 각 스텝마다 새롭게 계산되며, 정보 병목 현상을 해결한다.

## 3. Attention 메커니즘의 수식

1. 쿼리(Query): 현재 시점의 인식 상태 (예: 디코더에서 생성 중인 단어)  
2. 키(Key): 입력 시퀀스의 각 은닉 상태  
3. 값(Value): 입력 시퀀스의 각 은닉 상태  

### Attention 구조 이미지
![attention 구조](https://d.pr/i/f3W5Zg+)


### Attention Score 계산

$$
Score(Q, K) = Q \cdot K^T
$$

### Softmax를 통한 가중치 계산

$$
\alpha_i = \frac{\exp(Score(Q, K_i))}{\sum_j \exp(Score(Q, K_j))}
$$

### 최종 Context Vector 계산

$$
Context\ Vector = \sum_i \alpha_i V_i
$$


## 4. Attention 메커니즘의 장점

### 1. 정보 병목 현상 해결

- 고정된 컨텍스트 벡터 대신 가변적인 컨텍스트 벡터를 생성하여, 출력에 필요한 정보를 더 정확히 전달한다.

### 2. 장기 의존성 처리

- 시퀀스 전체 정보를 모두 고려하며, 중요한 부분에 집중하기 때문에 장기 의존성 문제를 완화한다.


### 3. 모듈화 및 확장성

- RNN뿐만 아니라 CNN 등 다른 모델에도 쉽게 결합 가능하며, 모델의 성능 향상을 가져온다.


### 4. 해석 가능성 (Interpretability)

- 가중치를 통해 모델이 어떤 입력에 집중했는지 확인할 수 있어, 결과를 해석하기 용이하다.


## Single-head Self-Attention
- 같은 입력 x에서 Q, K, V를 만든다.
- 문장 내부 토큰끼리 서로 참고한다.
- attention을 한 번 수행한다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# batch_size, seq_len, embedding_dim = (1,3,4)
x = torch.tensor([[[1.0, 0.0, 1.0, 0.0],
                   [0.0, 2.0, 0.0, 2.0],
                   [1.0, 1.0, 1.0, 1.0],
]])
print('x.shape : ',x.shape)

# Q : 내가 찾고 싶은 정보
# K : 내가 가진 정보의 이름표
# V : 실제 가져온 정보
# Q, K, V를 생성하는 선형층(학습 대상 가중치)
W_q = nn.Linear(4,4,bias=False)         # (in_feature,out_feature)
W_k = nn.Linear(4,4,bias=False)
W_v = nn.Linear(4,4,bias=False)

# 같은 x에서 Q,K,V를 만들지만 각각 다른 linear 층을 거치므로 서로 다른 관점의 벡터가 된다.
Q = W_q(x)
K = W_k(x)
V = W_v(x)

print("Q.shape : ",Q.shape)
print("K.shape : ",K.shape)
print("V.shape : ",V.shape)

# 1. attention score : Q @ K.T (Q,K의 유사도 계산)
attn_scores = torch.matmul(Q,K.transpose(-2,-1))        # (1, 3, 4) @ (1, 4, 3) matmul = 뒤에 두개 차원 내적 연산, 그러기 위해 transpose로 뒤에 두개 위치 변환
print('attn_score.shape : ',attn_scores.shape)          # (1, 3, 3) => 토큰 3개가 각각 다른 토큰 3개를 얼마나 참고할지 계산된 결과

# 2. attention wight : softmax(Q @ K.T) -> attention 분포/확률
# Q랑 K를 곱하면 값이 너무 커질 수 있고, 이때 softmax의 결과가 한쪽으로 쏠릴 수 있어서 값 안정화를 위해 스케일링(임베딩 차원의 루트 값)
attn_scores /= Q.size(-1) ** 0.5
attn_weight = F.softmax(attn_scores, dim=-1)    # 확률 값으로 변환
print('attn_weight.shape : ', attn_weight.shape)

# 3. attention value : softmax(Q @ K.T) @ V -> attention 분포의 가중합
attn_value = torch.matmul(attn_weight, V)   # (1,3,3) @ (1,3,4)
print('attn_value.shape : ', attn_value.shape)  # (1,3,4)

# 출력 shape이 입력 shape와 같은 이유 : 각 토큰이 다른 토큰 정보를 반영한 새로운 4차원 벡터로 변경되기 때문

x.shape :  torch.Size([1, 3, 4])
Q.shape :  torch.Size([1, 3, 4])
K.shape :  torch.Size([1, 3, 4])
V.shape :  torch.Size([1, 3, 4])
attn_score.shape :  torch.Size([1, 3, 3])
attn_weight.shape :  torch.Size([1, 3, 3])
attn_value.shape :  torch.Size([1, 3, 4])


## Multi-head Self-Attention
- 어텐션 연산을 num_head로 나누고 다른 관점으로 어텐션 연산을 수행한다

In [ ]:
# batch_size, seq_len, embedding_dim = (1,3,8)
x = torch.tensor([[[1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0],
                   [0.0, 2.0, 0.0, 2.0, 0.0, 2.0, 0.0, 2.0],
                   [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
]])
print('x.shape : ',x.shape)

B, T, C = x.shape

W_q = nn.Linear(C,C,bias=False)
W_k = nn.Linear(C,C,bias=False)
W_v = nn.Linear(C,C,bias=False)

Q = W_q(x)
K = W_k(x)
V = W_v(x)

print('Q.shape : ',Q.shape)
print('K.shape : ',K.shape)
print('V.shape : ',V.shape)

# 8차원 벡터를 4개의 head로 나누면 각 head는 2차원이 된다. (embedding 차원은 num_head로 나누어 떨어져야 함)
num_head = 4
head_dim = C // num_head

# 헤드 분할
# batch_size, seq_len, embedding_dim (1, 3, 8)
# -> batch_size, seq_len, num_head, head_dim (1, 3, 4, 2)
# ->  batch_size, num_head, seq_len, head_dim (1, 4, 3, 2)
Q_heads = Q.view(B,T,num_head,head_dim).transpose(1,2)
K_heads = K.view(B,T,num_head,head_dim).transpose(1,2)
V_heads = V.view(B,T,num_head,head_dim).transpose(1,2)

print('Q_heads.shape : ', Q_heads.shape)
print('K_heads.shape : ', K_heads.shape)
print('V_heads.shape : ', V_heads.shape)

# attention을 4개 head가 각각 수행할 수 있는 모양으로 변경 완료

# 1. attention score : Q @ K.T (Q @ K의 유사도 계산)
# (1, 4, 3, 2) @ (1, 4, 2, 3) -> (1, 4, 3, 3)
attn_scores = torch.matmul(Q_heads,K_heads.transpose(-2,-1))
print('attn_scores.shape : ',attn_scores.shape)
# head가 4개라는 것은 같은 문장을 4개의 관점으로 따로 보는 것이다.

# attention weight : softmax(Q @ K.T) -> softmax 분포/확률
attn_scores /= head_dim ** 0.5
attn_weight = F.softmax(attn_scores,dim=-1)
print('attn_weight.shape :',attn_weight.shape)

# 3. attention value : softmax(Q @ K.T) @ V -> attention 분포 가중합
# (1,4,3,3) @ (1,4,3,2) - > (1,4,3,2)
attn_value = torch.matmul(attn_weight, V_heads)
# 각 head가 자기만의 attention 결과를 만든다.

# 헤드 병합
output = attn_value.transpose(1,2)      # (1,3,4,2)
# contiguous : 호출 전 메모리 연속된 상태 변환
output = output.contiguous().view(B,T,C)
print('output.shape :',output.shape)
# 4개의 head 결과를 다시 하나의 8차원 벡터로 병합해서 변환

x.shape :  torch.Size([1, 3, 8])
Q.shape :  torch.Size([1, 3, 8])
K.shape :  torch.Size([1, 3, 8])
V.shape :  torch.Size([1, 3, 8])
Q_heads.shape :  torch.Size([1, 4, 3, 2])
K_heads.shape :  torch.Size([1, 4, 3, 2])
V_heads.shape :  torch.Size([1, 4, 3, 2])
attn_scores.shape :  torch.Size([1, 4, 3, 3])
attn_weight.shape : torch.Size([1, 4, 3, 3])
output.shape : torch.Size([1, 3, 8])
